# Week 6: Knowledge Representation: Propositional Logic

This week covers propositional syntax and semantics, truth tables, and truth-table enumeration for entailment. The notebook includes the observation-book cases shown in the deck.

**Source:** `PPTs/Week6.pptx`.


### Syntax and semantics
For propositions P and Q, `~P` negates P; `P & Q` is conjunction; `P | Q` is disjunction; `P >> Q` is implication; and `P == Q` is biconditional. Implication is false only when its antecedent is true and its consequent is false.

Entailment is checked over every model satisfying the knowledge base.

## Program 1 — Truth-table enumeration for propositional entailment

**Theory.** A proposition is true or false in a model. A knowledge base entails a query when every model that satisfies all knowledge-base sentences also satisfies the query. Enumeration checks all 2ⁿ assignments. The supported operators are NOT `~`, AND `&`, OR `|`, implication `>>` and biconditional `==`.

**Objective.** Enter a propositional knowledge base and query, then inspect the complete truth table and entailment result.

**Libraries.** Python standard library (`ast`, `itertools`).

**Algorithm.**
1. Read proposition names and formulas. 2. Enumerate all Boolean assignments. 3. Evaluate each sentence in each assignment. 4. Mark models where the knowledge base is true. 5. Report entailment only if the query is true in all such models.

**Complexity.** O(2ⁿ × L) time and O(2ⁿ) table storage, where L is the total formula size.

The code below keeps `input()` so you can provide your own values. The final lines run a sample interaction through the same prompts; edit `SAMPLE_INPUTS` to try another case, or call the main function directly for a live terminal run.

**Expected output and explanation.** The output reports the computed result for the entered values.


In [1]:
from contextlib import contextmanager
import builtins

@contextmanager
def sample_input(values):
    original_input = builtins.input
    remaining = iter(map(str, values))
    def demo_input(prompt=''):
        print(prompt, end='')
        try:
            value = next(remaining)
        except StopIteration as exc:
            raise RuntimeError('Add more values to SAMPLE_INPUTS for this run.') from exc
        print(value)
        return value
    builtins.input = demo_input
    try:
        yield
    finally:
        builtins.input = original_input

import ast, itertools

def pl_true(expression,model):
    node=ast.parse(expression,mode='eval').body
    def evaluate(n):
        if isinstance(n,ast.Name): return model[n.id]
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,ast.Invert): return not evaluate(n.operand)
        if isinstance(n,ast.BinOp):
            left,right=evaluate(n.left),evaluate(n.right)
            if isinstance(n.op,ast.BitAnd): return left and right
            if isinstance(n.op,ast.BitOr): return left or right
            if isinstance(n.op,ast.RShift): return (not left) or right
        if isinstance(n,ast.Compare) and len(n.ops)==1 and isinstance(n.ops[0],ast.Eq):
            return evaluate(n.left)==evaluate(n.comparators[0])
        raise ValueError('Use proposition names with ~, &, |, >> and == only.')
    return evaluate(node)

def truth_table(kb,query,symbols):
    rows=[]; entails=True
    for values in itertools.product((False,True),repeat=len(symbols)):
        model=dict(zip(symbols,values)); kb_value=all(pl_true(s,model) for s in kb); q_value=pl_true(query,model)
        rows.append((values,kb_value,q_value))
        if kb_value and not q_value: entails=False
    return rows,entails

def main():
    try:
        n=int(input('Number of propositions: ')); symbols=input('Proposition names separated by spaces: ').split()
        m=int(input('Number of knowledge-base sentences: '))
    except ValueError: print('Counts must be integers.'); return
    if n<1 or n!=len(symbols) or len(set(symbols))!=n or any(not s.isidentifier() for s in symbols):
        print('Provide a positive count and distinct identifier names.'); return
    if m<1: print('Enter at least one knowledge-base sentence.'); return
    kb=[input(f'KB sentence {i+1}: ') for i in range(m)]; query=input('Query: ')
    try: rows,result=truth_table(kb,query,symbols)
    except (SyntaxError,KeyError,ValueError) as exc: print('Invalid formula:',exc); return
    print('\n'+'\t'.join(symbols+['KB','Query']))
    for values,kb_value,q_value in rows: print('\t'.join(str(int(v)) for v in (*values,kb_value,q_value)))
    print('\nKB entails query:',result)


SAMPLE_INPUTS = ['2', 'P Q', '2', 'P >> Q', 'P', 'Q']
with sample_input(SAMPLE_INPUTS):
    main()


Number of propositions: 2
Proposition names separated by spaces: P Q
Number of knowledge-base sentences: 2
KB sentence 1: P >> Q
KB sentence 2: P
Query: Q

P	Q	KB	Query
0	0	0	0
0	1	0	1
1	0	0	0
1	1	1	1

KB entails query: True


## Program 2 — Observation-book entailment questions

**Theory.** The Week 6 deck lists three knowledge bases and asks truth-table entailment questions for each. We reuse the same truth-table engine to check every requested query instead of implementing a separate logic procedure.

**Objective.** Show whether the Week 6 observation-book queries follow from their stated knowledge bases.

**Libraries.** Python standard library; uses the previous program’s `truth_table` function.

**Algorithm.**
1. Select one of the three slide problems. 2. Build its knowledge-base formulas. 3. List all requested queries. 4. Enumerate models and report whether each query is entailed.

**Complexity.** O(q × 2ⁿ × L) time for q queries; O(2ⁿ) temporary table storage per query.

The code below keeps `input()` so you can provide your own values. The final lines run a sample interaction through the same prompts; edit `SAMPLE_INPUTS` to try another case, or call the main function directly for a live terminal run.

**Expected output and explanation.** The output reports the computed result for the entered values.


In [2]:
from contextlib import contextmanager
import builtins

@contextmanager
def sample_input(values):
    original_input = builtins.input
    remaining = iter(map(str, values))
    def demo_input(prompt=''):
        print(prompt, end='')
        try:
            value = next(remaining)
        except StopIteration as exc:
            raise RuntimeError('Add more values to SAMPLE_INPUTS for this run.') from exc
        print(value)
        return value
    builtins.input = demo_input
    try:
        yield
    finally:
        builtins.input = original_input

import ast, itertools

def pl_true(expression,model):
    node=ast.parse(expression,mode='eval').body
    def evaluate(n):
        if isinstance(n,ast.Name): return model[n.id]
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,ast.Invert): return not evaluate(n.operand)
        if isinstance(n,ast.BinOp):
            left,right=evaluate(n.left),evaluate(n.right)
            if isinstance(n.op,ast.BitAnd): return left and right
            if isinstance(n.op,ast.BitOr): return left or right
            if isinstance(n.op,ast.RShift): return (not left) or right
        if isinstance(n,ast.Compare) and len(n.ops)==1 and isinstance(n.ops[0],ast.Eq):
            return evaluate(n.left)==evaluate(n.comparators[0])
        raise ValueError('Use proposition names with ~, &, |, >> and == only.')
    return evaluate(node)

def truth_table(kb,query,symbols):
    rows=[]; entails=True
    for values in itertools.product((False,True),repeat=len(symbols)):
        model=dict(zip(symbols,values)); kb_value=all(pl_true(s,model) for s in kb); q_value=pl_true(query,model)
        rows.append((values,kb_value,q_value))
        if kb_value and not q_value: entails=False
    return rows,entails

def main():
    problems={
      '1':(['P','Q','R'],['Q >> P','P >> ~Q','Q | R'],['R','R >> P','Q >> R']),
      '2':(['S','T'],['~(S | T)'],['S & T','T | ~T']),
      '3':(['R','S'],['(R >> S) & (S >> R)'],['R == S','(R & S) | (~R & ~S)'])}
    print('Problem 1: KB=(Q→P) & (P→¬Q) & (Q∨R)')
    print('Problem 2: KB=¬(S∨T)')
    print('Problem 3: KB=(R→S) & (S→R)')
    choice=input('Choose problem (1/2/3): ').strip()
    if choice not in problems: print('Choose 1, 2 or 3.'); return
    symbols,kb,queries=problems[choice]
    for query in queries:
        _,result=truth_table(kb,query,symbols)
        print(f'KB entails {query}: {result}')


SAMPLE_INPUTS = ['1']
with sample_input(SAMPLE_INPUTS):
    main()


Problem 1: KB=(Q→P) & (P→¬Q) & (Q∨R)
Problem 2: KB=¬(S∨T)
Problem 3: KB=(R→S) & (S→R)
Choose problem (1/2/3): 1
KB entails R: True
KB entails R >> P: False
KB entails Q >> R: True
